# Eval: Gemini (Google AI Studio) — **tuần tự + delay**

Notebook riêng cho Gemini qua OpenAI-compatible endpoint.

| | |
|---|---|
| Endpoint | `https://generativelanguage.googleapis.com/v1beta/openai/` |
| Token | `GEMINI_API_KEY` trong `doan/.env` (hoặc gán `TOKEN`) |
| Parallel | **`MAX_CONCURRENCY = 1`** (tuần tự) |
| Pace | **`REQUEST_DELAY_SEC`** — nghỉ giữa 2 request |

**MODE** (ablation):

| MODE | Schema | Thinking | Examples |
|---|---|---|---|
| ORIG | Không | Không | Có |
| S | Có | Không | Có |
| T | Không | Có | Có |
| ST | Có | Có | Có |
| ST-E | Có | Có | Không |

Closed-model coverage gợi ý **ORIG + ST** trước (`configs/model_coverage.yaml`).

- Resume mặc định: câu đủ `n_samples` trong `scores.jsonl` → bỏ qua.
- Output: `results/<model>/<MODE>/` — dùng chung với `20_compare_summary.ipynb`.
- Free / low tier Gemini dễ 429 → tăng `REQUEST_DELAY_SEC` (vd. 12–15s). `gemini-3.6-flash` free ≈ **5 RPM / 20 RPD**.
- **Thinking control (Gemini 3.x):** không tắt được (`none` → 400). Dùng `REASONING_EFFORT`:
  - ORIG / S → tự gửi `minimal` (thấp nhất)
  - T / ST / ST-E → lấy `REASONING_EFFORT` (`low` | `medium` | `high`)


In [1]:
# === INPUTS ===
MODEL = "gemini-3.6-flash"  # slug Google OpenAI-compat
TOKEN = ""  # trống → GEMINI_API_KEY từ doan/.env
BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
MODE = "ORIG"  # ORIG | S | T | ST | ST-E

# Số sample / câu (None = experiment.yaml, mặc định 20)
N_SAMPLES = 5

# --- Pace (gemini-3.6-flash free tier ≈ 5 RPM, ~20 RPD) ---
MAX_CONCURRENCY = 1          # tuần tự; đừng tăng
REQUEST_DELAY_SEC = 6     # ★ DELAY giữa 2 lần gọi (giây). 5 RPM → ≥12
RATE_LIMIT_RETRIES = 6       # 429 thì đợi rồi thử lại

# Closed model → paper temperature 1.5
TEMPERATURE = 1.5

# Thinking control (Gemini 3.x KHÔNG tắt được — none → 400):
#   ORIG / S  → code tự gửi reasoning_effort="minimal"
#   T/ST/ST-E → dùng giá trị này: "low" | "medium" | "high"
REASONING_EFFORT = "minimal"

In [2]:
%pip install -q openai pyyaml tqdm
%pip install -q -e ../..


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import sys
from pathlib import Path

HERE = Path.cwd().resolve()
REPO = None
for c in [HERE, *HERE.parents]:
    if (c / "configs" / "experiment.yaml").exists():
        REPO = c
        break
    if (c / "doan" / "configs" / "experiment.yaml").exists():
        REPO = c / "doan"
        break
assert REPO is not None and (REPO / "configs" / "experiment.yaml").exists(), (
    "Chạy notebook từ trong repo doan/"
)
sys.path.insert(0, str(REPO / "src"))
print("REPO:", REPO)

env_file = REPO / ".env"
if env_file.exists():
    for line in env_file.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        k, v = k.strip(), v.strip().strip('"').strip("'")
        if k and k not in os.environ:
            os.environ[k] = v
    print("Loaded .env:", env_file)
else:
    print("No .env at", env_file)

if not TOKEN:
    TOKEN = (
        os.environ.get("GEMINI_API_KEY")
        or os.environ.get("GOOGLE_API_KEY")
        or os.environ.get("OPENAI_API_KEY")
        or ""
    )
if not TOKEN:
    raise RuntimeError(
        "TOKEN trống — thêm GEMINI_API_KEY=... vào doan/.env hoặc gán TOKEN"
    )

assert MAX_CONCURRENCY == 1, "Notebook này cố ý tuần tự — giữ MAX_CONCURRENCY = 1"
assert REQUEST_DELAY_SEC is not None and float(REQUEST_DELAY_SEC) >= 0

n_calls_est = 50 * int(N_SAMPLES or 20)
pace = float(REQUEST_DELAY_SEC)
print("MODEL:", MODEL)
print("BASE_URL:", BASE_URL)
print("MODE:", MODE)
print("N_SAMPLES:", N_SAMPLES)
print("MAX_CONCURRENCY:", MAX_CONCURRENCY, "(sequential)")
print(f"DELAY giữa các lần gọi: {pace}s  (~{60 / pace:.1f} RPM)")
print(f"Ước lượng floor time (chỉ delay, chưa tính latency API): {n_calls_est * pace / 60:.1f} phút / {n_calls_est} calls")
print("RATE_LIMIT_RETRIES:", RATE_LIMIT_RETRIES)
print("TEMPERATURE:", TEMPERATURE)
print("REASONING_EFFORT:", REASONING_EFFORT)
print("TOKEN set:", bool(TOKEN), "(len=", len(TOKEN), ")")

REPO: /Users/nguyenkz/Documents/code/CS2202.CH202/doan
Loaded .env: /Users/nguyenkz/Documents/code/CS2202.CH202/doan/.env
MODEL: gemini-3.6-flash
BASE_URL: https://generativelanguage.googleapis.com/v1beta/openai/
MODE: ORIG
N_SAMPLES: 5
MAX_CONCURRENCY: 1 (sequential)
DELAY giữa các lần gọi: 6.0s  (~10.0 RPM)
Ước lượng floor time (chỉ delay, chưa tính latency API): 25.0 phút / 250 calls
RATE_LIMIT_RETRIES: 6
TEMPERATURE: 1.5
REASONING_EFFORT: minimal
TOKEN set: True (len= 39 )


In [ ]:
from plausibility_eval.run_eval import run_evaluation

result = run_evaluation(
    model=MODEL,
    token=TOKEN,
    base_url=BASE_URL,
    mode=MODE,
    repo=REPO,
    resume=True,
    n_samples_override=N_SAMPLES,
    max_concurrency=MAX_CONCURRENCY,
    request_delay_sec=REQUEST_DELAY_SEC,
    rate_limit_retries=RATE_LIMIT_RETRIES,
    temperature_override=TEMPERATURE,
    reasoning_effort=REASONING_EFFORT,
    openrouter_providers=None,
    openrouter_allow_fallbacks=None,
)
print("out_dir:", result["out_dir"])
print("metrics:", result["metrics"])

/Users/nguyenkz/Documents/code/CS2202.CH202/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[eval] model=gemini-3.6-flash mode=ORIG provider=gemini_openai_compat
       sentences=50 n_samples=5 → 250 API calls (full)
       resume=True skip_done=1 todo=49 remaining_calls=241
       temp=1.5 max_tokens=512 concurrency=1 reasoning=minimal
       request_delay_sec=6.0 rate_limit_retries=6
       providers=None out=/Users/nguyenkz/Documents/code/CS2202.CH202/doan/results/gemini-3.6-flash/ORIG
[eval] 429 s1_global#4 — đợi 65s rồi retry (1/6)                
[eval] 429 s1_global#4 — đợi 65s rồi retry (2/6)                
[eval] 429 s1_global#4 — đợi 65s rồi retry (3/6)                
[eval] 429 s1_global#4 — đợi 65s rồi retry (4/6)                
[eval] 429 s1_global#4 — đợi 65s rồi retry (5/6)                
[eval] 429 s1_global#4 — đợi 65s rồi retry (6/6)                
[eval] ERROR s1_global#4: Error code: 429 - [{'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https

In [ ]:
# Quick evidence check
import json
from pathlib import Path

out = Path(result["out_dir"])
calls = list((out / "calls").glob("*.json"))
print("n call files:", len(calls))
if calls:
    sample = json.loads(calls[0].read_text(encoding="utf-8"))
    print("keys:", sorted(sample.keys()))
    print("provider:", sample.get("provider"))
    print("trace_id:", sample.get("trace_id"))
    print("usage:", sample.get("usage"))
    print("has request:", bool(sample.get("request")))
    print("has output_text:", bool(sample.get("output_text")))
    print("reasoning_text is None?:", sample.get("reasoning_text") is None)
print("scores.jsonl lines:", sum(1 for _ in open(out / "scores.jsonl", encoding="utf-8")))
assert "cost_usd" not in (out / "scores.jsonl").read_text(encoding="utf-8")[:2000]
print("OK: no cost_usd in scores head")

## Gợi ý chạy

1. Thêm `GEMINI_API_KEY=...` vào `doan/.env`.
2. Smoke: `N_SAMPLES = 2`, `MODE = "ORIG"`, chạy cell eval.
3. Full: `N_SAMPLES = 20`, đổi `MODE` → chạy lại (resume giữ progress).
4. Nếu 429 nhiều: tăng `REQUEST_DELAY_SEC` (12–15) hoặc giảm `N_SAMPLES` tạm.
5. Summary: `notebooks/20_compare_summary.ipynb`.